FAISS basics 
1. from_documents()
2. from_embeddings()
3. similarity_search
4. similarity_search_by_vector
5. similarity_search_with_score
6. similarity_search_with_score_by_vector
5. similarity_search_with_relevance_scores<br>

Topics:
ANN and ENN
Storage - only vector

Index types:
1. IndexFlatL2
2. IndexFlatIP
3. IVF (Inverted file index)
4. HNSW (Hierarchical Navigable Small World)
5. PQ (Product Quantization)

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
import os
from dotenv import load_dotenv

#load key for hugging face
load_dotenv()
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

#1. load the doc
loader = PyPDFLoader("/Users/sanvijanvi/Documents/SJ_HomeWork/attention.pdf")
pdfDoc = loader.load()

#2. Chunking docs
seperators = ["\n\n","\n"," ",""]
pdfSplitter = RecursiveCharacterTextSplitter(separators=seperators,chunk_size = 1000, chunk_overlap = 100)
pdfChunks = pdfSplitter.split_documents(pdfDoc)

#3. Embeddings
pdfEmbeder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")



In [ ]:
from langchain_community.vectorstores import FAISS

vectorStore = FAISS.from_documents(documents=pdfChunks, embedding=pdfEmbeder)

# similarity search.
answerSS = vectorStore.similarity_search(query="What is attention mechanism?",k=1)
# print(answerSS)
# print(len(answerSS))
# print(answerSS[0].page_content)

# similarity_search_by_vector

answerSSV = vectorStore.similarity_search_by_vector(embedding=pdfEmbeder.embed_query("What is attention mechanism?"),k=2)
print(answerSSV[0].page_content)

In [ ]:
vectorStore.index

In [ ]:
print(len(pdfChunks))
print(vectorStore.index.ntotal)


In [ ]:
ansScore = vectorStore.similarity_search_with_score(query = "scaled dot-product attention softmax queries keys value")
for elem in ansScore:
    print(f"score : {elem[1]}")

In [ ]:
ansRelScore = vectorStore.similarity_search_with_relevance_scores(query = "scaled dot-product attention softmax queries keys value",score_threshold=0.6)
for elem in ansRelScore:
    print(f"score : {elem[1]}")


In [ ]:
print(ansRelScore[0][0].metadata)

# MMR

In [ ]:
ansMMR = vectorStore.max_marginal_relevance_search(query="scaled dot-product attention softmax queries keys value",k=2, fetch_k=10,lambda_mult=0.7)
print(len(ansMMR),type(ansMMR[0]))

#Persistence

In [ ]:
vectorStore.save_local("/Users/sanvijanvi/AI_Workspace/ML_DL_Projects/genAI_langchain_projects/langchain_langraph/riseInfo/langchain")

In [ ]:
vectorstore = FAISS.load_local(folder_path="/Users/sanvijanvi/AI_Workspace/ML_DL_Projects/genAI_langchain_projects/langchain_langraph/riseInfo/langchain",allow_dangerous_deserialization=True,embeddings=pdfEmbeder)
print(vectorstore.index.ntotal)

In [ ]:
filter_data = {'page': 4}
ansMMR = vectorstore.max_marginal_relevance_search(query="scaled dot-product attention softmax queries keys value",k=5, fetch_k=30,lambda_mult=0.7,filter=filter_data)
print(len(ansMMR),type(ansMMR[0]))

In [ ]:
from langchain_core.documents import Document
documents = [
    "Machine learning is a subset of artificial intelligence.",
    "Deep learning uses neural networks.",
    "Python is used for data science.",
    "Cooking requires different ingredients.",
    "Football is a popular sport."
]

docList = [Document(page_content=text) for text in documents]
print(type(docList))


In [ ]:
docIds = vectorstore.add_documents(documents=docList)
print(type(docIds),docIds[0],len(docIds))


In [ ]:
vectorstore.index.ntotal

In [ ]:
vectorstore.index_to_docstore_id

In [ ]:
embeddDim = len(pdfEmbeder.embed_query("What is attention?"))
type(embeddDim)

In [ ]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore

index = faiss.IndexHNSWFlat(embeddDim,32)
vectorHNSW = FAISS(embedding_function=pdfEmbeder, index = index, docstore= InMemoryDocstore(),index_to_docstore_id={} )
vectorHNSW.add_documents(pdfChunks)
vectorHNSW.index.ntotal


# Chroma DB

In [ ]:
from langchain_community.vectorstores import Chroma

vectorChroma = Chroma(collection_name="DBExample",persist_directory="./chroma_db1",embedding_function=pdfEmbeder)
vectorChroma.add_documents(pdfChunks)

del vectorChroma._collection
del vectorChroma

In [ ]:
ids = [str(num) for num in range(len(pdfChunks))]
print(ids)

vectorChroma = Chroma.from_documents(documents=pdfChunks,collection_name="DBExample",persist_directory="./chroma_db",embedding=pdfEmbeder,ids=ids)

In [ ]:
collection = vectorChroma._collection
print(collection.name)
print(collection.count())

In [ ]:
filter_data1 = {"page":{"$gt":5}}
filter_data2 = {"$or":[{"page":8},{"page":9}]}
chromaMMR = vectorChroma.max_marginal_relevance_search(query="scaled dot-product attention softmax queries keys value", k = 3, fetch_k=10,
                                 filter= filter_data2)


for elem in chromaMMR:
    print(elem.metadata["page"])

# Filtering :
Chroma has native metadata filtering/query capabilities, while LangChain's FAISS vector store implements metadata filtering around the FAISS index and associated document store. Therefore, the exact filtering behavior/API depends on the LangChain FAISS implementation.<br>
1. $eq   equal
2. $ne   not equal
3. $gt   greater than
4. $gte  greater than or equal
5. $lt   less than
6. $lte  less than or equal
7. $in   matches any value in a list
8. $nin  doesn't match values in a list
9. $and
10. $or
11. $contains

# collection
Chroma <br>
├── from_documents()<br>
├── from_texts()<br>
├── collection<br>
├── collection.get() ---> include, where,where_docuemnts,ids<br> 
├── collection.query()  --> query_texts, n_results, include, where,where_document<br>
├── collection.add()<br>
├── collection.delete()<br>
├── collection.update()<br>
├── collection_name<br>
├── persist_directory<br>
└── reopening/loading existing collections<br>

In [ ]:
collection.name

In [ ]:
collection.get().keys()

In [ ]:
embedding = collection.get(include=["embeddings","documents"],ids=["10"])
print(len(embedding))

In [ ]:
len(collection.get())

In [ ]:
embedding = collection.get(where={"$or":[{"page":3},{"page":5}]})["documents"]
len(embedding)

In [ ]:
embedding = collection.get(where_document={"$contains":"formula"})["documents"]
print(len(embedding))

In [ ]:
answers = collection.query(query_texts=["scaled dot-product attention softmax queries keys value","What is attention mechanism?"], n_results=5)
print(answers["ids"])
print(type(answers))

In [ ]:
vectorChroma1 = Chroma(collection_name="DBExample",persist_directory="./chroma_db1",embedding_function=pdfEmbeder)

In [ ]:
collection1 = vectorChroma1._collection
print(collection1.count())
print(collection1.get()["ids"][-1])

In [ ]:
documents = [
    "Machine learning is a subset of artificial intelligence.",
    "Deep learning uses neural networks.",
    "Python is used for data science.",
    "Cooking requires different ingredients.",
    "Football is a popular sport."
]

docListCnt = collection1.count()
docLen = len(documents)
maxLen = docListCnt + docLen

print(docListCnt,docLen,maxLen)

In [ ]:
from langchain_community.docstore.document import Document

docs = [Document(page_content=text) for text in documents]

In [ ]:
ids1 = [str(num) for num in range(docListCnt,maxLen)]
ids1

In [ ]:
collection1.add(ids =ids1,documents=documents)

In [ ]:
collection1.count()

In [ ]:
collection1.get(ids=["298"])

In [ ]:
collection1.delete(ids=["298"])

In [ ]:
collection1.get(ids=["298"])

In [ ]:
collection1.count()

In [ ]:
collection1.get(ids=["297"])

In [ ]:
collection1.update(ids=["297"],documents=["Hi"])

In [ ]:
collection1.get(ids=["297"],include= ['metadatas', 'documents'])

In [ ]:
res = collection1.get(ids=["297"], include=["embeddings"])
print(res["included"])
print(res["embeddings"])

In [ ]:
collection1.count()